# RAG（検索拡張生成）の基礎


## Weave のセットアップとトレースの確認


In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

# APIキーの先頭3文字が表示されれば、環境変数に設定できています
print(os.environ["WANDB_API_KEY"][:3])
assert os.environ["WANDB_API_KEY"]

# WANDB_PROJECTの値が「training-ai-agent-dev-」だとエラーになります
print(os.environ["WANDB_PROJECT"])
assert os.environ["WANDB_PROJECT"] != "training-ai-agent-dev-"

In [ ]:
# Weaveの初期化
import os

import weave

weave.init(os.environ["WANDB_PROJECT"])

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model


@weave.op()
def generate_recipe(dish: str) -> str:
    prompt = [
        SystemMessage(content="ユーザーが入力した料理のレシピを考えてください。"),
        HumanMessage(content=f"料理名: {dish}"),
    ]
    model = init_chat_model(
        model="gpt-5-nano",
        model_provider="openai",
        reasoning_effort="minimal",
    )
    ai_message = model.invoke(prompt)
    return ai_message.content


recipe = generate_recipe("カレー")
print(recipe)

# ここでWeaveのトレースを確認してください


# RAG (検索拡張生成) の基礎


## LangChain での RAG の実装をステップバイステップで動かそう


### 検索対象ファイルのダウンロード

**利用資料・ライセンス等に関する注意事項**

本ハンズオンでは、受講者が各自、以下の公開資料を公式サイトからダウンロードし、RAG（Retrieval-Augmented Generation）の検索対象データとして利用します。
ハンズオン中に、各資料に対してテキスト抽出、チャンク分割、メタデータ付与、ベクトル化、検索インデックス化等の処理を行う場合があります。

1. OWASP Top 10 for LLM Applications 2025

   発行：
   OWASP GenAI Security Project / OWASP Foundation

   Version / Date：
   Version 2025 / November 18, 2024

   Source：
   https://genai.owasp.org/resource/owasp-top-10-for-llm-applications-2025/

   PDF：
   https://genai.owasp.org/download/43299/?tmstv=1731900559

   License：
   Creative Commons Attribution-ShareAlike 4.0 International（CC BY-SA 4.0）
   https://creativecommons.org/licenses/by-sa/4.0/

2. AI セーフティに関する評価観点ガイド（第1.10版）

   発行：
   AI セーフティ・インスティテュート（AISI）

   日付：
   令和7年3月28日

   出典：
   AI セーフティ・インスティテュート（AISI）ウェブサイト

   PDF：
   https://aisi.go.jp/assets/pdf/ai_safety_eval_v1.10_ja.pdf

   掲載ページ：
   https://aisi.go.jp/output/output_information/250328_1/

加工について：
本ハンズオンでは、受講者自身の操作により、上記資料からのテキスト抽出、分割、ベクトル化、検索インデックス化等を実施します。
これらの処理結果、検索結果表示、および生成AIによる回答は、元資料そのものではなく、ハンズオン環境上で加工・生成されたものです。

OWASP資料について：
「OWASP Top 10 for LLM Applications 2025」由来の内容は、CC BY-SA 4.0に基づいて利用します。
OWASP資料由来のテキスト、抜粋、要約、チャンク、検索用データ、その他の加工物を再配布または共有する場合は、CC BY-SA 4.0の条件に従ってください。
具体的には、出典表示、ライセンス表示、変更・加工を行った旨の表示、同一ライセンス条件での共有等が必要となる場合があります。

AISI資料について：
「AI セーフティに関する評価観点ガイド（第1.10版）」は、AISIが公開している資料です。
本ハンズオンでは、出典を明示したうえで、研修目的での参照、テキスト抽出、分割、検索対象化、生成AIによる要約・回答生成に利用します。
AISI資料を直接引用する場合は、引用箇所を明確にし、可能な限り原文のまま扱ってください。
一部を改変、要約、再構成、またはRAG回答として生成する場合は、元資料をもとに加工・生成されたものであることを明示してください。
AISI資料のPDF本体、抽出テキスト、チャンク、検索インデックス等を外部に再配布・公開する場合は、AISI/IPAの公表する利用条件や案内を確認してください。

有償研修に関する注意：
本ハンズオンで利用する上記資料はいずれも公開資料です。
本研修の有償部分は、講師による解説、演習設計、実装支援、ハンズオン環境、サポート等の付加価値であり、公開資料そのものを有償販売するものではありません。

免責・非推奨表示：
本ハンズオン環境、本研修、検索結果、および生成AIによる回答は、研修実施者が作成・実装するものです。
OWASP、OWASP Foundation、AI セーフティ・インスティテュート（AISI）、または情報処理推進機構（IPA）が、本研修、本ハンズオン環境、検索結果、生成AIによる回答を作成、監修、保証、推奨、または承認するものではありません。

第三者権利物・商標等について：
各資料に含まれる第三者権利物、商標、ロゴ、シンボルマーク、外部リンク先のコンテンツ等は、各資料本文のライセンスや本注意事項とは別の権利・条件の対象となる場合があります。
OWASP、AISI、IPA等の名称、ロゴ、商標を、本研修が公式・認定・推奨されたものであると誤認させる態様で使用しないでください。


In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
import shutil

# ダウンロードしたいPDFのURLとファイル名のリスト
pdf_list = [
    {
        "url": "https://genai.owasp.org/download/43299/?tmstv=1731900559",
        "filename": "OWASP_Top_10_for_LLM_Applications_2025.pdf",
    },
    {
        "url": "https://aisi.go.jp/assets/pdf/ai_safety_eval_v1.10_ja.pdf",
        "filename": "AIセーフティに関する評価観点ガイド（第1.10版）.pdf",
    },
]

docs_dir = Path("../tmp/docs/")
docs_dir.mkdir(parents=True, exist_ok=True)

for pdf in pdf_list:
    out = docs_dir / pdf["filename"]
    req = Request(
        pdf["url"],
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/pdf,*/*",
        },
    )
    with urlopen(req, timeout=60) as res, out.open("wb") as f:
        shutil.copyfileobj(res, f)
    print(f"Downloaded: {out}")

### Document loader


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# 指定ディレクトリ以下のPDFファイルをPyPDFLoaderで読み込み、デフォルトではページで分割
loader = DirectoryLoader(
    path="../tmp/docs/",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

raw_docs = loader.load()
print(len(raw_docs))

### Document transformer


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# チャンキング
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

docs = text_splitter.split_documents(raw_docs)
print(len(docs))

### Embedding model


In [ ]:
from langchain.embeddings import init_embeddings

query = "プロンプトインジェクションとは"

# 埋め込みモデルの初期化
embeddings = init_embeddings(model="text-embedding-3-small", provider="openai")

# ベクトル化
vector = embeddings.embed_query(query)

print(len(vector))
print(vector)

### Vector store


In [ ]:
# 注意:
# このセルの処理は、大勢が同時に実行するとレートリミットのエラーになる可能性があります
# もしもエラーになった場合は、少し時間をおいてもう一度実行してください

import os
from langchain_chroma import Chroma

# 大量のトレースを保存しないよう、一時的にWeaveを無効化
os.environ["WEAVE_DISABLED"] = "true"

# Chroma（OSSのベクトルデータベース）に登録、指定した埋め込みモデルを使って内部でベクトルに変換される
vector_store = Chroma(embedding_function=embeddings, persist_directory="../tmp/chroma")
vector_store.reset_collection()
vector_store.add_documents(docs)

os.environ["WEAVE_DISABLED"] = "false"


### Retriever


In [ ]:
query = "プロンプトインジェクションとは"

# 検索インターフェース作成
retriever = vector_store.as_retriever()

# 検索実行
context_docs = retriever.invoke(query)
print(f"len = {len(context_docs)}")

for i, doc in enumerate(context_docs):
    print(
        "-" * 10 + f" {i + 1}/{len(context_docs)}: {doc.metadata['source']} " + "-" * 10
    )
    print(doc.page_content)


### LangChain を使った RAG の実装


In [ ]:
import weave
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model


_prompt_template = '''
以下の文脈だけを踏まえて質問に回答してください。

文脈: """
{context}
"""

質問: {question}
'''


@weave.op
def invoke_rag(query: str) -> str:
    documents = retriever.invoke(query)
    prompt_text = _prompt_template.format(question=query, context=documents)

    model = init_chat_model(
        model="gpt-5.4-nano",
        model_provider="openai",
        reasoning_effort="none",
    )
    ai_message = model.invoke([HumanMessage(content=prompt_text)])
    return ai_message.content


query = "プロンプトインジェクションとは"
output = invoke_rag(query)
print(output)